# Per-rank read statistics: 640 ranks

Validates and summarizes the per-rank fio JSONs from the MPI read benchmarks
on container `fio-10tib-fs16g` (20 nodes x 32 ranks, one rank per 16 GiB file,
bs=2m, iodepth=16, one iteration). These runs use `SIZE=15g`, so each rank
reads 15 GiB of its file (some files are not fully 16 GiB, see the write-job
notes).

Sections, one per read pattern:
- **Sequential read** — job 8658418 (`PATTERN=read`)
- **Random read** — job 8658861 (`PATTERN=randread`), to be added when it completes

**Validation** — every rank file must parse, report `error=0`, all ranks must
read the same volume and that volume must fit in the shortest file
(16 GiB - 28 MiB, the largest write deficit), with no short/dropped IOs,
and ranks 0..639 must all be present.

**Statistics** — aggregate bandwidth two ways plus per-rank distributions:
- **Wall-clock aggregate**: total bytes / (last rank end - first rank start), using
  each job's `job_start` + `runtime`. The honest number, bounded by stragglers.
- **Sum of per-rank `bw_bytes`**: optimistic upper bound; each rank's average only
  applies while that rank was running, so skew is ignored.

fio units: `read.bw_bytes` is B/s (`bw` is the same in KiB/s), `read.runtime` is ms,
`lat_ns.mean` is the mean completion latency in ns.

In [ ]:
import json
import re
import statistics as st
from pathlib import Path

import matplotlib.pyplot as plt

EXPECT_RANKS = 640
PPN = 32
NNODES = EXPECT_RANKS // PPN
# 39 of the 640 files are short of the full 16 GiB; the largest deficit is
# 28 MiB (large.494, see write_logs/fio-randwrite.note). A uniform per-rank
# read volume is only valid if it fits in the shortest file.
MAX_READ_BYTES = 16 * 1024**3 - 28 * 1024**2
GiB = 1024**3

## Sequential read (job 8658418)

`PATTERN=read`: each rank streams its 15 GiB sequentially, once.

In [ ]:
RUN_NAME = "read_fio-10tib-fs16g_n20_ppn32_bs2m_iod16_8658418"

# Run folder sits next to this notebook.
RESULT_DIR = Path(RUN_NAME)
files = sorted(RESULT_DIR.glob("*.json"))
print(f"{len(files)} rank files in {RESULT_DIR}")

### Validation: did every rank read the same volume, within the shortest file?

In [ ]:
problems = []
rows = []   # one dict per rank

for fp in files:
    txt = fp.read_text()
    try:
        d = json.loads(txt[txt.index("{"):])   # skip any non-JSON header line
    except (ValueError, json.JSONDecodeError) as e:
        problems.append(f"{fp.name}: unparseable JSON ({e})")
        continue
    rank = int(re.search(r"rank(\d+)", fp.name).group(1))
    if len(d["jobs"]) != 1:
        problems.append(f"{fp.name}: {len(d['jobs'])} jobs, expected 1")
    j = d["jobs"][0]
    r = j["read"]
    if j["error"] != 0:
        problems.append(f"{fp.name}: fio error={j['error']}")
    if r["short_ios"] or r["drop_ios"]:
        problems.append(f"{fp.name}: short_ios={r['short_ios']} drop_ios={r['drop_ios']}")
    if j["jobname"] != f"large.{rank}":
        problems.append(f"{fp.name}: jobname {j['jobname']} != large.{rank}")
    rows.append({
        "rank": rank,
        "io_bytes": r["io_bytes"],
        "runtime_ms": r["runtime"],
        "bw_bytes": r["bw_bytes"],
        "lat_mean_ns": r["lat_ns"]["mean"],
        "start_ms": j.get("job_start", d["timestamp_ms"]),
    })

# Every rank must read the same volume, and that volume must be no larger
# than the shortest file (16 GiB - 28 MiB), or the short files read less
# data than the rest and the per-rank numbers are not comparable.
volumes = sorted({r["io_bytes"] for r in rows})
if len(volumes) > 1:
    problems.append(f"ranks read differing volumes: {[f'{v/GiB:.3f} GiB' for v in volumes]}")
if volumes and volumes[-1] > MAX_READ_BYTES:
    problems.append(f"per-rank read {volumes[-1]} B exceeds the shortest file "
                    f"({MAX_READ_BYTES} B = 16 GiB - 28 MiB)")

ranks_seen = {r["rank"] for r in rows}
missing = set(range(EXPECT_RANKS)) - ranks_seen
if missing:
    problems.append(f"missing ranks: {sorted(missing)}")

if problems:
    print(f"PROBLEMS ({len(problems)}):")
    for p in problems:
        print(" -", p)
else:
    print(f"All reads successful: {len(rows)} ranks, error=0, uniform "
          f"io_bytes={volumes[0]/GiB:.2f} GiB/rank (< 16 GiB - 28 MiB), "
          f"no short/dropped IOs, ranks 0-{EXPECT_RANKS-1} complete.")

### Aggregate bandwidth

In [ ]:
total_bytes = sum(r["io_bytes"] for r in rows)
starts = [r["start_ms"] for r in rows]
ends = [r["start_ms"] + r["runtime_ms"] for r in rows]
wall_s = (max(ends) - min(starts)) / 1000
sum_bw = sum(r["bw_bytes"] for r in rows)

print(f"Total read              : {total_bytes/1024**4:.2f} TiB in {len(rows)} ranks")
print(f"Wall clock              : {wall_s:.1f} s (first rank start -> last rank end)")
print(f"Aggregate bandwidth     : {total_bytes/wall_s/GiB:.1f} GiB/s = {total_bytes/wall_s/1e9:.1f} GB/s")
print(f"Sum of per-rank bw      : {sum_bw/GiB:.1f} GiB/s (upper bound, ignores skew)")

### Per-rank distributions

In [ ]:
runtimes_s = [r["runtime_ms"] / 1000 for r in rows]
bws_gib = [r["bw_bytes"] / GiB for r in rows]
lats_ms = [r["lat_mean_ns"] / 1e6 for r in rows]

def pctl(vals, p):
    s = sorted(vals)
    return s[min(len(s) - 1, int(round(p / 100 * (len(s) - 1))))]

hdr = f"{'metric':<22} {'min':>8} {'p50':>8} {'mean':>8} {'p99':>8} {'max':>8} {'stdev':>8}"
print(hdr)
print("-" * len(hdr))
for name, vals in [("runtime (s)", runtimes_s),
                   ("bandwidth (GiB/s)", bws_gib),
                   ("mean latency (ms)", lats_ms)]:
    print(f"{name:<22} {min(vals):>8.2f} {st.median(vals):>8.2f} {st.mean(vals):>8.2f} "
          f"{pctl(vals, 99):>8.2f} {max(vals):>8.2f} {st.stdev(vals):>8.2f}")

### Stragglers

The slowest ranks bound the wall-clock aggregate: the median rank finishes in
~12 s but the tail stretches the pass to ~102 s, which is why the wall-clock
aggregate sits far below the summed per-rank bandwidths.

In [ ]:
slowest = sorted(rows, key=lambda r: -r["runtime_ms"])[:10]
print(f"{'rank':>5} {'node':>5} {'runtime s':>10} {'bw GiB/s':>9} {'mean lat ms':>12}")
for r in slowest:
    print(f"{r['rank']:>5} {r['rank']//32:>5} {r['runtime_ms']/1000:>10.1f} "
          f"{r['bw_bytes']/GiB:>9.2f} {r['lat_mean_ns']/1e6:>12.1f}")

### Per-node bandwidth distribution

One box per node (32 per-rank bandwidths each). If the straggler tail were
caused by a few slow nodes, their whole boxes would sit low; fliers hanging
below otherwise-normal boxes mean slow *ranks* scattered across nodes.

In [ ]:
# One box per node: 32 per-rank bandwidths each (node n hosts ranks n*32..n*32+31).
bw_by_node = [[] for _ in range(NNODES)]
for r in rows:
    bw_by_node[r["rank"] // PPN].append(r["bw_bytes"] / GiB)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.boxplot(
    bw_by_node, positions=range(NNODES), widths=0.55, patch_artist=True,
    boxprops=dict(facecolor="#cde2fb", edgecolor="#1c5cab", linewidth=1),
    whiskerprops=dict(color="#1c5cab", linewidth=1),
    capprops=dict(color="#1c5cab", linewidth=1),
    medianprops=dict(color="#1c5cab", linewidth=1.6),
    flierprops=dict(marker="o", markersize=4, markerfacecolor="#86b6ef",
                    markeredgecolor="none", alpha=0.85),
)
overall_med = st.median(r["bw_bytes"] / GiB for r in rows)
ax.axhline(overall_med, color="#8a8a8a", linewidth=1, linestyle="--", zorder=0)
ax.annotate(f"overall median {overall_med:.2f} GiB/s", xy=(4, overall_med),
            ha="center", va="bottom", fontsize=9, color="#555555")

ax.set_xticks(range(NNODES))
ax.set_xlabel("node")
ax.set_ylabel("per-rank read bandwidth (GiB/s)")
ax.set_title(f"Sequential read: per-rank bandwidth by node ({PPN} ranks/node)",
             fontsize=11, color="#333333")
ax.yaxis.grid(True, color="#e4e4e4", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c9c9c9")
ax.tick_params(colors="#555555")
plt.tight_layout()
plt.show()

## Random read (job 8658861)

`PATTERN=randread`, same geometry (640 ranks / 20 nodes, 15 GiB per rank).
To be added when the job completes: mirror the sequential-read cells above
with `RUN_NAME = "randread_fio-10tib-fs16g_n20_ppn32_bs2m_iod16_8658861"`.